# BTC Orderbook LSTM v2 — Microstructure Direction Prediction

**Changes vs v1:**
- **Model**: LSTM + multi-head self-attention over all timesteps → 2.8M params (was 212K)
- **Scale**: HIDDEN_DIM 128→512, NUM_LAYERS 2→3, SEQ_LEN 32→64 (16-minute window)
- **Training**: AMP/fp16 (3-4× throughput), focal loss (replaces weighted CE), warmup LR
- **Memory**: sequences stored as float16 → ~50% CPU RAM reduction
- **Fix**: confidence sweep now correctly counts only directional predictions as trades

## Setup
1. Runtime → Change runtime type → **T4 GPU**
2. Run all cells in order

In [ ]:
# ── Cell 1: Install and imports ───────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'scikit-learn', 'pandas', 'numpy', 'matplotlib'], check=True)

import os, glob, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
#!apt install sshpass -y && sshpass -p XXXXXXXXXX sftp -r -o "StrictHostKeyChecking no" scpuser@155.207.120.2:btc_data.tar.xz .

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
#!ls
#!mv btc_data.tar.xz /content/drive/MyDrive/
#!ls /content/drive/MyDrive/ -l

In [ ]:
#drive.flush_and_unmount()

In [ ]:
# ── Cell 2: Mount Google Drive + config ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!cd ~ && mkdir -p btc_lstm_v2
!cd ~ && tar xJf /content/drive/MyDrive/btc_data.tar.xz

drive.flush_and_unmount()

# ── CONFIGURE THESE ───────────────────────────────────────────────────────
DATA_DIR   = '/root/btc_data'
OUTPUT_DIR = '/root/btc_lstm_v2'

THRESHOLD  = 50.0    # USD — same as best XGBoost run
HORIZON    = 8       # bars ahead = 2 minutes
SEQ_LEN    = 64      # 16 minutes of context (was 32)
BATCH_SIZE = 8192    # 4× larger; fp16 keeps VRAM in check
EPOCHS     = 60
WARMUP     = 5       # linear warmup epochs before cosine decay
LR         = 3e-4    # lower LR for bigger model
HIDDEN_DIM = 512     # was 128
NUM_LAYERS = 3       # was 2
NHEAD      = 8       # attention heads (HIDDEN_DIM must be divisible by NHEAD)
DROPOUT    = 0.25
FOCAL_GAMMA = 2.0   # focal loss gamma; 0 = standard weighted CE
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
# ─────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Data dir:   {DATA_DIR}')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# ── Cell 3: Feature pipeline (unchanged from v1) ──────────────────────────
SEPARATOR = ';'
WINDOW_SEC = 15

COLS_NEEDED_BASE = {
    'spot_datetime', 'future_datetime', 'spot_timestamp', 'future_timestamp',
    'future_bid_close', 'future_ask_close',
    'future_bid_open', 'future_bid_max', 'future_bid_min',
    'future_bid_median', 'future_ask_median',
    'future_spread_open', 'future_spread_max',
    'future_buy_qty', 'future_sell_qty',
    'future_buy_samples', 'future_sell_samples',
    'future_buy_vwap', 'future_sell_vwap', 'future_price_diff',
    'future_bid_liq_0.0_median', 'future_ask_liq_0.0_median',
    'spot_buy_qty', 'spot_sell_qty',
    'opt_open_interest_sample', 'opt_funding_rate_sample',
    'opt_est_funding_rate_sample', 'opt_remaining_time_sample',
    'opt_long_force_exit_qty_sum', 'opt_short_force_exit_qty_sum',
    'future_first_trade_side', 'future_last_trade_side',
    'future_largest_trade_qty', 'future_largest_trade_side',
    'future_buy_count_early', 'future_buy_count_late',
    'future_sell_count_early', 'future_sell_count_late',
    'future_buy_qty_early', 'future_buy_qty_late',
    'future_sell_qty_early', 'future_sell_qty_late',
}
COLS_NEEDED_LIQ_DEEP  = [
    'future_bid_liq_0.04_median', 'future_ask_liq_0.04_median',
    'future_bid_liq_0.05_median', 'future_ask_liq_0.05_median',
    'future_bid_liq_0.06_median', 'future_ask_liq_0.06_median',
]
COLS_NEEDED_LIQ_TOTAL = [
    'future_bid_liq_0.1_median',  'future_ask_liq_0.1_median',
    'future_bid_liq_0.2_median',  'future_ask_liq_0.2_median',
    'future_bid_liq_0.3_median',  'future_ask_liq_0.3_median',
    'future_bid_liq_0.4_median',  'future_ask_liq_0.4_median',
]

ALL_FEATURES = [
    'book_imbalance', 'flow_imbalance', 'momentum', 'book_imbal_deep',
    'flow_imbal_roll4', 'flow_imbal_roll8', 'book_imbal_roll4', 'book_imbal_roll8',
    'vwap_spread', 'liq_flag', 'stochastic', 'spread_expansion', 'sample_imbalance',
    'flow_agreement', 'oi_change', 'size_imbalance', 'liq_conc_bid', 'liq_conc_ask',
    'hour_sin', 'hour_cos', 'minute_sin', 'minute_cos',
    'near_funding', 'funding_pressure', 'vol_norm',
    'trade_side_open', 'trade_side_close', 'trade_side_momentum',
    'largest_trade_side', 'largest_trade_rel',
    'buy_accel', 'sell_accel', 'flow_accel', 'buy_count_accel', 'late_imbalance',
]


def load_files(files):
    all_needed = COLS_NEEDED_BASE | set(COLS_NEEDED_LIQ_DEEP) | set(COLS_NEEDED_LIQ_TOTAL)
    frames = []
    for path in files:
        try:
            hdr = pd.read_csv(path, sep=SEPARATOR, nrows=0)
            usecols = sorted(all_needed & set(hdr.columns))
            if 'spot_datetime' not in usecols:
                print(f'  SKIP {os.path.basename(path)}: missing spot_datetime')
                continue
            df = pd.read_csv(path, sep=SEPARATOR, usecols=usecols, low_memory=False)
            df = df[df['spot_datetime'] != 'spot_datetime']
            frames.append(df)
            print(f'  Loaded {os.path.basename(path):45s}  {len(df):>7,} rows  ({len(usecols)} cols)')
        except Exception as e:
            print(f'  SKIP {os.path.basename(path)}: {e}')
    if not frames:
        raise RuntimeError('No valid CSV files found.')
    return pd.concat(frames, ignore_index=True)


def prepare(df, threshold, horizon):
    df['dt'] = pd.to_datetime(df['spot_datetime'], errors='coerce')
    df = df.dropna(subset=['dt']).sort_values('dt').reset_index(drop=True)
    n_before = len(df)
    df = df.drop_duplicates(subset='dt', keep='first').reset_index(drop=True)
    if len(df) < n_before:
        print(f'  Dropped {n_before - len(df)} duplicate timestamps')

    skip = {'spot_datetime', 'future_datetime', 'dt'}
    for col in df.columns:
        if col not in skip:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    for col in [c for c in df.columns if c.startswith('opt_')]:
        df[col] = df[col].replace(-1, np.nan)

    if 'future_bid_close' in df.columns and 'future_ask_close' in df.columns:
        df['close_price'] = (df['future_bid_close'] + df['future_ask_close']) / 2.0
    else:
        df['close_price'] = (df['future_bid_median'] + df['future_ask_median']) / 2.0

    next_close   = df['close_price'].shift(-horizon)
    delta        = next_close - df['close_price']
    df['target'] = np.where(delta > threshold, 1, np.where(delta < -threshold, -1, 0))
    df['delta_usd'] = delta

    eps = 1e-9
    df['flow_imbalance'] = ((df['future_buy_qty'] - df['future_sell_qty']) /
                            (df['future_buy_qty'] + df['future_sell_qty'] + eps))
    bid0 = df['future_bid_liq_0.0_median']
    ask0 = df['future_ask_liq_0.0_median']
    df['book_imbalance'] = (bid0 - ask0) / (bid0 + ask0 + eps)
    df['momentum'] = df['future_price_diff']

    deep_col = None
    for lvl in ['0.05', '0.04', '0.06']:
        b, a = f'future_bid_liq_{lvl}_median', f'future_ask_liq_{lvl}_median'
        if b in df.columns and a in df.columns:
            deep_col = (b, a); break
    df['book_imbal_deep'] = ((df[deep_col[0]] - df[deep_col[1]]) /
                             (df[deep_col[0]] + df[deep_col[1]] + eps)) if deep_col else np.nan

    df['flow_imbal_roll4'] = df['flow_imbalance'].rolling(4, min_periods=2).mean()
    df['flow_imbal_roll8'] = df['flow_imbalance'].rolling(8, min_periods=4).mean()
    df['book_imbal_roll4'] = df['book_imbalance'].rolling(4, min_periods=2).mean()
    df['book_imbal_roll8'] = df['book_imbalance'].rolling(8, min_periods=4).mean()

    df['vwap_spread'] = (df['future_buy_vwap'] - df['future_sell_vwap']
                         if 'future_buy_vwap' in df.columns else np.nan)
    liq_long  = df.get('opt_long_force_exit_qty_sum',  pd.Series(0, index=df.index))
    liq_short = df.get('opt_short_force_exit_qty_sum', pd.Series(0, index=df.index))
    df['liq_flag'] = ((liq_long + liq_short) > 0).astype(float)

    if all(c in df.columns for c in ['future_bid_max', 'future_bid_min', 'future_bid_close']):
        rng = df['future_bid_max'] - df['future_bid_min']
        df['stochastic'] = np.where(rng > 0, (df['future_bid_close'] - df['future_bid_min']) / rng, 0.5)
    else:
        df['stochastic'] = np.nan

    df['spread_expansion'] = (df['future_spread_max'] - df['future_spread_open']
                              if all(c in df.columns for c in ['future_spread_max', 'future_spread_open'])
                              else np.nan)

    if all(c in df.columns for c in ['future_buy_samples', 'future_sell_samples']):
        bs, ss = df['future_buy_samples'], df['future_sell_samples']
        df['sample_imbalance'] = (bs - ss) / (bs + ss + eps)
    else:
        df['sample_imbalance'] = np.nan

    if all(c in df.columns for c in ['spot_buy_qty', 'spot_sell_qty']):
        spot_flow = ((df['spot_buy_qty'] - df['spot_sell_qty']) /
                     (df['spot_buy_qty'] + df['spot_sell_qty'] + eps))
        df['flow_agreement'] = df['flow_imbalance'] * spot_flow
    else:
        df['flow_agreement'] = np.nan

    df['oi_change'] = df['opt_open_interest_sample'].diff() if 'opt_open_interest_sample' in df.columns else np.nan

    if all(c in df.columns for c in ['future_buy_samples', 'future_sell_samples',
                                      'future_buy_qty', 'future_sell_qty']):
        df['size_imbalance'] = (df['future_buy_qty']  / (df['future_buy_samples']  + eps) -
                                df['future_sell_qty'] / (df['future_sell_samples'] + eps))
    else:
        df['size_imbalance'] = np.nan

    sub_cols = ['future_first_trade_side', 'future_last_trade_side',
                'future_largest_trade_qty', 'future_largest_trade_side',
                'future_buy_count_early', 'future_buy_count_late',
                'future_sell_count_early', 'future_sell_count_late',
                'future_buy_qty_early', 'future_buy_qty_late',
                'future_sell_qty_early', 'future_sell_qty_late']
    if all(c in df.columns for c in sub_cols):
        bqe, bql = df['future_buy_qty_early'],  df['future_buy_qty_late']
        sqe, sql = df['future_sell_qty_early'],  df['future_sell_qty_late']
        fts, lts = df['future_first_trade_side'], df['future_last_trade_side']
        lq,  ls  = df['future_largest_trade_qty'], df['future_largest_trade_side']
        bce, bcl = df['future_buy_count_early'], df['future_buy_count_late']
        total_vol = bqe + bql + sqe + sql
        df['trade_side_open']     = fts
        df['trade_side_close']    = lts
        df['trade_side_momentum'] = lts - fts
        df['largest_trade_side']  = ls
        df['largest_trade_rel']   = lq / (total_vol + eps)
        df['buy_accel']           = bql - bqe
        df['sell_accel']          = sql - sqe
        df['flow_accel']          = (bql - sql) - (bqe - sqe)
        df['buy_count_accel']     = bcl - bce
        df['late_imbalance']      = (bql - sql) / (bql + sql + eps)
    else:
        for c in ['trade_side_open','trade_side_close','trade_side_momentum',
                  'largest_trade_side','largest_trade_rel','buy_accel','sell_accel',
                  'flow_accel','buy_count_accel','late_imbalance']:
            df[c] = np.nan

    deep_bid = next((c for c in ['future_bid_liq_0.4_median','future_bid_liq_0.3_median',
                                  'future_bid_liq_0.2_median','future_bid_liq_0.1_median']
                     if c in df.columns), None)
    deep_ask = next((c for c in ['future_ask_liq_0.4_median','future_ask_liq_0.3_median',
                                  'future_ask_liq_0.2_median','future_ask_liq_0.1_median']
                     if c in df.columns), None)
    df['liq_conc_bid'] = (df['future_bid_liq_0.0_median'] / (df[deep_bid] + eps)
                          if deep_bid else np.nan)
    df['liq_conc_ask'] = (df['future_ask_liq_0.0_median'] / (df[deep_ask] + eps)
                          if deep_ask else np.nan)

    hour, minute = df['dt'].dt.hour, df['dt'].dt.minute
    df['hour_sin']   = np.sin(2 * np.pi * hour   / 24)
    df['hour_cos']   = np.cos(2 * np.pi * hour   / 24)
    df['minute_sin'] = np.sin(2 * np.pi * minute / 60)
    df['minute_cos'] = np.cos(2 * np.pi * minute / 60)

    secs = (hour * 3600 + minute * 60 + df['dt'].dt.second) % (8 * 3600)
    df['near_funding'] = ((8 * 3600 - secs) < 900).astype(float)
    if 'opt_funding_rate_sample' in df.columns:
        funding = df['opt_funding_rate_sample'].replace(-1, np.nan).fillna(0)
        df['funding_pressure'] = funding * df['near_funding']
    else:
        df['funding_pressure'] = np.nan

    df['volatility'] = df['close_price'].diff().abs().rolling(16, min_periods=4).mean()
    df['vol_norm']   = (df['volatility'] /
                        df['volatility'].rolling(5760, min_periods=96).mean()
                       ).replace([np.inf, -np.inf], np.nan)

    df = df.iloc[:-horizon].copy()
    return df

print('Feature pipeline loaded.')

In [ ]:
# ── Cell 4: Load and prepare data ────────────────────────────────────────
files = sorted(glob.glob(os.path.join(DATA_DIR, '*.csv')) +
               glob.glob(os.path.join(DATA_DIR, '*.csv.gz')))
print(f'Found {len(files)} files')

raw = load_files(files)
print(f'\nTotal rows loaded: {len(raw):,}')

df = prepare(raw, THRESHOLD, HORIZON)
del raw

features = [f for f in ALL_FEATURES if f in df.columns and df[f].notna().sum() > 100]
print(f'\nFeatures available: {len(features)}/35')
print(f'Date range: {df["dt"].iloc[0]} → {df["dt"].iloc[-1]}')
print(f'Total rows after prepare: {len(df):,}')

vc = df['target'].value_counts().sort_index()
for k, v in vc.items():
    label = {-1: 'DOWN', 0: 'FLAT', 1: 'UP'}[k]
    print(f'  {label}: {v:,} ({v/len(df)*100:.1f}%)')

In [ ]:
# ── Cell 5: Scale + build sequences (float16 → ~50% RAM vs v1) ───────────
#
# v1 stored sequences as float32: with SEQ_LEN=64 that would be ~10 GB.
# We scale in float32 (for precision), then cast to float16 for storage.
# DataLoader converts back to float32 on the fly before GPU transfer.

clean = df[features + ['target', 'close_price', 'dt']].dropna().reset_index(drop=True)
print(f'Clean rows: {len(clean):,}  ({len(clean)/len(df)*100:.1f}% of prepared data)')

X_all    = clean[features].values.astype(np.float32)
y_all    = clean['target'].values
price_all = clean['close_price'].values
dt_all    = clean['dt'].values

n         = len(clean)
train_end = int(n * TRAIN_FRAC)
val_end   = int(n * (TRAIN_FRAC + VAL_FRAC))

scaler = StandardScaler()
X_all[:train_end] = scaler.fit_transform(X_all[:train_end])
X_all[train_end:] = scaler.transform(X_all[train_end:])
print(f'Scaler fitted on {train_end:,} training rows')

# Cast to float16 BEFORE building sequences to save RAM
X_all_f16 = X_all.astype(np.float16)
del X_all

def make_sequences_f16(X, y, seq_len):
    N, F = X.shape
    seqs   = np.lib.stride_tricks.sliding_window_view(X, (seq_len, F)).reshape(-1, seq_len, F)
    labels = y[seq_len - 1:]
    return seqs.copy(), labels  # .copy() materialises the view so GC can free the original

X_train_seq, y_train = make_sequences_f16(X_all_f16[:train_end],        y_all[:train_end],        SEQ_LEN)
X_val_seq,   y_val   = make_sequences_f16(X_all_f16[train_end:val_end], y_all[train_end:val_end], SEQ_LEN)
X_test_seq,  y_test  = make_sequences_f16(X_all_f16[val_end:],          y_all[val_end:],          SEQ_LEN)
price_test            = price_all[val_end + SEQ_LEN - 1:]
dt_test               = dt_all[val_end + SEQ_LEN - 1:]
del X_all_f16

seq_mem = (X_train_seq.nbytes + X_val_seq.nbytes + X_test_seq.nbytes) / 1e9
print(f'\nSequences built (seq_len={SEQ_LEN} bars = {SEQ_LEN*WINDOW_SEC}s):')
print(f'  Train: {len(X_train_seq):,}  Val: {len(X_val_seq):,}  Test: {len(X_test_seq):,}')
print(f'  Input shape: {X_train_seq.shape}  dtype=float16')
print(f'  Total sequence RAM: {seq_mem:.2f} GB')

In [ ]:
# ── Cell 6: Dataset and DataLoader ───────────────────────────────────────

class SeqDataset(Dataset):
    def __init__(self, X, y):
        # X stored as float16 tensor; .float() in __getitem__ is nearly free
        self.X = torch.from_numpy(X)                        # float16
        self.y = torch.tensor(y + 1, dtype=torch.long)      # 0/1/2
    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i].float(), self.y[i]

train_ds = SeqDataset(X_train_seq, y_train)
val_ds   = SeqDataset(X_val_seq,   y_val)
test_ds  = SeqDataset(X_test_seq,  y_test)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True, persistent_workers=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

# Softer class weights: square-root of inverse frequency (less aggressive than 'balanced')
freqs = np.bincount(y_train + 1) / len(y_train)   # [p_down, p_flat, p_up]
sqrt_weights = 1.0 / np.sqrt(freqs)
sqrt_weights /= sqrt_weights.mean()                # normalise so mean=1
class_weights = torch.tensor(sqrt_weights, dtype=torch.float32).to(device)
print(f'Class weights (sqrt): DOWN={sqrt_weights[0]:.2f}  FLAT={sqrt_weights[1]:.2f}  UP={sqrt_weights[2]:.2f}')
print(f'(v1 balanced were: DOWN≈1.80  FLAT≈0.53  UP≈1.83)')

In [ ]:
# ── Cell 7: Model — LSTM + multi-head self-attention ─────────────────────
#
# Architecture:
#   LayerNorm → 3-layer LSTM → multi-head self-attention (residual) → classifier
#
# The attention layer attends over all SEQ_LEN timestep outputs, not just the
# last one. This lets the model weight which bars in the window matter most.
# Bidirectionality is safe here: the whole sequence is historical.

class BtcLSTMAttn(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, nhead, dropout, num_classes=3):
        super().__init__()
        self.input_norm = nn.LayerNorm(input_dim)
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.attn      = nn.MultiheadAttention(hidden_dim, nhead, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(hidden_dim)
        # Feed-forward block after attention
        self.ff = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
        )
        self.ff_norm  = nn.LayerNorm(hidden_dim)
        self.dropout  = nn.Dropout(dropout)
        self.fc       = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x: (batch, seq_len, features)
        x = self.input_norm(x)
        h, _ = self.lstm(x)                           # (batch, seq_len, hidden)
        attn_out, _ = self.attn(h, h, h)             # self-attention
        h = self.attn_norm(h + attn_out)             # residual
        h = self.ff_norm(h + self.ff(h))             # FF residual
        last = h[:, -1, :]                            # use last position
        return self.fc(self.dropout(last))


model = BtcLSTMAttn(
    input_dim=len(features),
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    nhead=NHEAD,
    dropout=DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')
print(model)
if device.type == 'cuda':
    print(f'GPU memory after model init: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
# ── Cell 8: Training — AMP + focal loss + warmup ──────────────────────────
#
# Focal loss: down-weights easy examples (large pt), focuses capacity on
# hard ambiguous bars. gamma=2 is standard; set FOCAL_GAMMA=0 for plain CE.

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


def make_scheduler(optimizer, warmup_epochs, total_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


criterion = FocalLoss(gamma=FOCAL_GAMMA, weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = make_scheduler(optimizer, WARMUP, EPOCHS)
scaler_amp = GradScaler()  # AMP gradient scaler

best_val_loss = float('inf')
best_state    = None
patience      = 10
wait          = 0

train_losses, val_losses, val_accs = [], [], []

for epoch in range(1, EPOCHS + 1):
    # ── Train with AMP
    model.train()
    running_loss = 0.0
    for X_b, y_b in train_dl:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        with autocast():
            loss = criterion(model(X_b), y_b)
        scaler_amp.scale(loss).backward()
        scaler_amp.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler_amp.step(optimizer)
        scaler_amp.update()
        running_loss += loss.item() * len(X_b)
    train_loss = running_loss / len(train_ds)

    # ── Validate
    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_b, y_b in val_dl:
            X_b, y_b = X_b.to(device), y_b.to(device)
            with autocast():
                logits = model(X_b)
            val_loss += criterion(logits.float(), y_b).item() * len(X_b)
            correct  += (logits.float().argmax(1) == y_b).sum().item()
            total    += len(y_b)
    val_loss /= len(val_ds)
    val_acc   = correct / total
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        wait          = 0
        flag = '  ← best'
    else:
        wait += 1
        flag  = f'  (patience {wait}/{patience})'

    if epoch % 5 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]['lr']
        if device.type == 'cuda':
            vram = torch.cuda.max_memory_allocated() / 1e9
            print(f'Epoch {epoch:3d}  train={train_loss:.4f}  val={val_loss:.4f}  '
                  f'val_acc={val_acc*100:.2f}%  lr={lr_now:.2e}  VRAM={vram:.1f}GB{flag}')
        else:
            print(f'Epoch {epoch:3d}  train={train_loss:.4f}  val={val_loss:.4f}  '
                  f'val_acc={val_acc*100:.2f}%  lr={lr_now:.2e}{flag}')

    if wait >= patience:
        print(f'\nEarly stopping at epoch {epoch}')
        break

model.load_state_dict(best_state)
print(f'\nBest val loss: {best_val_loss:.4f}')

In [ ]:
# ── Cell 9: Evaluation and backtest ──────────────────────────────────────

le_inv = {0: -1, 1: 0, 2: 1}

model.eval()
all_logits, all_labels = [], []
with torch.no_grad():
    for X_b, y_b in test_dl:
        with autocast():
            logits = model(X_b.to(device)).float().cpu()
        all_logits.append(logits)
        all_labels.append(y_b)

logits_all = torch.cat(all_logits)
proba_all  = torch.softmax(logits_all, dim=1).numpy()
pred_enc   = logits_all.argmax(1).numpy()
preds      = np.array([le_inv[p] for p in pred_enc])
confidence = proba_all.max(axis=1)

labels_flat = torch.cat(all_labels).numpy()
y_test_orig = np.array([le_inv[l] for l in labels_flat])

acc       = (preds == y_test_orig).mean()
dummy_acc = max(np.bincount(labels_flat) / len(labels_flat))
print(f'Test accuracy : {acc*100:.2f}%')
print(f'Dummy baseline: {dummy_acc*100:.2f}%')
print(f'Lift          : {(acc - dummy_acc)*100:+.2f}%')

trade_pnl = np.where(preds[:-1] ==  1,  np.diff(price_test),
            np.where(preds[:-1] == -1, -np.diff(price_test), 0.0))

bnh_pnl   = price_test[-1] - price_test[0]
strat_pnl = trade_pnl.sum()
n_trades  = (preds[:-1] != 0).sum()
n_long    = (preds[:-1] ==  1).sum()
n_short   = (preds[:-1] == -1).sum()
trades_pnl = trade_pnl[preds[:-1] != 0]
win_rate  = (trades_pnl > 0).mean() if len(trades_pnl) > 0 else 0
avg_win   = trades_pnl[trades_pnl > 0].mean() if (trades_pnl > 0).any() else 0
avg_loss  = trades_pnl[trades_pnl < 0].mean() if (trades_pnl < 0).any() else 0

avg_price = price_test.mean()
maker_fee = avg_price * 0.0004
taker_fee = avg_price * 0.0010

print(f'\n{"─"*60}')
print(f'BACKTEST (test set, zero fees)')
print(f'{"─"*60}')
print(f'Test period  : {pd.Timestamp(dt_test[0])} → {pd.Timestamp(dt_test[-1])}')
print(f'BTC price    : ${price_test[0]:,.0f} → ${price_test[-1]:,.0f}  (Δ${price_test[-1]-price_test[0]:+,.0f})')
print(f'Total signals: {n_trades:,}  ({n_trades/len(preds)*100:.1f}% of bars)')
print(f'Long / Short : {n_long:,} / {n_short:,}')
print(f'Strategy P&L : ${strat_pnl:+,.2f}  (vs B&H ${bnh_pnl:+,.2f})')
print(f'Win rate     : {win_rate*100:.2f}%  over {n_trades:,} trades')
print(f'Avg win      : ${avg_win:.2f}  |  Avg loss: ${avg_loss:.2f}')
print(f'Expected P&L/trade: ${win_rate*avg_win + (1-win_rate)*avg_loss:.2f}')
print(f'Maker fee/trade  : ${maker_fee:.2f}')
print(f'After maker fees : ${strat_pnl - n_trades*maker_fee:+,.2f}')
print(f'After taker fees : ${strat_pnl - n_trades*taker_fee:+,.2f}')

# Fixed confidence sweep: only directional predictions count as trades
print(f'\nConfidence threshold sweep (maker fee = ${maker_fee:.0f}/trade):')
print(f'  {"Threshold":>10}  {"Trades":>7}  {"WinRate":>8}  {"AvgP&L":>8}  {"After maker":>12}')
for thresh in [0.33, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]:
    mask = (confidence[:-1] >= thresh) & (preds[:-1] != 0)  # directional only
    nt   = mask.sum()
    if nt == 0:
        print(f'  {thresh:>10.0%}  {0:>7,}  {"":>8}  {"":>8}  {"":>12}')
        continue
    tp    = trade_pnl[mask]
    wr    = (tp > 0).mean()
    avg   = tp.mean()
    after = tp.sum() - nt * maker_fee
    ok    = '✓' if after > 0 else '✗'
    print(f'  {thresh:>10.0%}  {nt:>7,}  {wr*100:>7.2f}%  {avg:>+8.2f}  {after:>+12,.0f}  {ok}')

try:
    from scipy.stats import binomtest
    n_wins = int(n_trades * win_rate)
    p = binomtest(n_wins, n_trades, 0.5, alternative='greater').pvalue
    print(f'\nBinomial test: win rate {win_rate*100:.2f}% over {n_trades:,} trades → p={p:.4f} '
          f'{"✓ significant" if p < 0.05 else "✗ not significant"}')
except:
    pass

In [ ]:
# ── Cell 9b: Extended statistics ─────────────────────────────────────────
# All variables here are also used by Cell 10 (plots), so run this first.
from sklearn.metrics import classification_report, confusion_matrix as sk_cm

# ── 1. Classification report
print('=' * 68)
print('CLASSIFICATION REPORT')
print('=' * 68)
print(classification_report(y_test_orig, preds, target_names=['DOWN', 'FLAT', 'UP'], digits=4))

# ── 2. Confusion matrix
cm      = sk_cm(y_test_orig, preds, labels=[-1, 0, 1])
cm_pct  = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9) * 100
print('CONFUSION MATRIX  (rows = true, cols = predicted)')
print(f'  {" ":>10}  {"pred DOWN":>16}  {"pred FLAT":>16}  {"pred UP":>16}')
for lbl, row, row_pct in zip(['true DOWN', 'true FLAT', 'true UP'], cm, cm_pct):
    cells = '  '.join(f'{v:>8,} ({p:>5.1f}%)' for v, p in zip(row, row_pct))
    print(f'  {lbl:>10}  {cells}')

# ── 3. Equity-curve metrics
cum_pnl     = np.cumsum(trade_pnl)
running_max = np.maximum.accumulate(np.maximum(cum_pnl, 0))  # floor at 0 so early DD is meaningful
drawdown    = cum_pnl - running_max
max_dd      = drawdown.min()
max_dd_idx  = drawdown.argmin()
peak_at_dd  = running_max[max_dd_idx]
max_dd_pct  = (max_dd / peak_at_dd * 100) if peak_at_dd > 0 else float('nan')

bars_per_year = 365 * 86400 / WINDOW_SEC
ret_per_bar   = trade_pnl / (price_test[:-1] + 1e-9)
sharpe        = ret_per_bar.mean() / (ret_per_bar.std() + 1e-12) * np.sqrt(bars_per_year)
calmar        = strat_pnl / abs(max_dd) if max_dd < 0 else float('inf')

wins_sum      = trades_pnl[trades_pnl > 0].sum()  if (trades_pnl > 0).any() else 0.0
loss_sum      = abs(trades_pnl[trades_pnl < 0].sum()) if (trades_pnl < 0).any() else 1e-9
profit_factor = wins_sum / loss_sum

# ── 4. Win/loss streaks
max_win_streak, max_loss_streak, cur_w, cur_l = 0, 0, 0, 0
for win in (trades_pnl > 0):
    if win:  cur_w += 1; cur_l  = 0
    else:    cur_l += 1; cur_w  = 0
    max_win_streak  = max(max_win_streak,  cur_w)
    max_loss_streak = max(max_loss_streak, cur_l)

print(f'\n{"=" * 68}')
print('EQUITY METRICS')
print(f'{"=" * 68}')
print(f'  Profit factor    : {profit_factor:.3f}  (>1 = gross edge)')
print(f'  Max drawdown     : ${max_dd:,.2f}  ({max_dd_pct:.1f}% of peak equity)')
print(f'  Sharpe (ann.)    : {sharpe:.3f}  (>1 useful, >2 strong)')
print(f'  Calmar ratio     : {calmar:.3f}  (total return / |max_dd|)')
print(f'  Max win streak   : {max_win_streak}')
print(f'  Max loss streak  : {max_loss_streak}')
print(f'  Long / Short     : {n_long:,} / {n_short:,}  ({n_long/(n_trades+1e-9)*100:.1f}% long bias)')

# ── 5. P&L by signal direction
print(f'\n{"=" * 68}')
print('P&L BY SIGNAL DIRECTION')
print(f'  {"Dir":>6}  {"Trades":>7}  {"WinRate":>8}  {"Avg P&L":>9}  {"Sum P&L":>12}')
for sig, lbl in [(1, 'LONG'), (-1, 'SHORT')]:
    m = preds[:-1] == sig
    if m.sum() == 0: continue
    tp_sig = trade_pnl[m]
    print(f'  {lbl:>6}  {m.sum():>7,}  {(tp_sig>0).mean()*100:>7.2f}%  {tp_sig.mean():>+9.2f}  {tp_sig.sum():>+12.2f}')

# ── 6. P&L by hour of day (UTC)
dt_trade_idx = pd.DatetimeIndex(dt_test[:-1])[preds[:-1] != 0]
pnl_series   = pd.Series(trades_pnl.copy(), index=dt_trade_idx)
grp_hour     = pnl_series.groupby(dt_trade_idx.hour)
pnl_by_hour  = grp_hour.sum()   # used by Cell 10 plots

print(f'\n{"=" * 68}')
print('P&L BY HOUR (UTC)')
print(f'  {"Hour":>4}  {"Trades":>7}  {"WinRate":>8}  {"Sum P&L":>12}  {"Avg P&L":>9}')
for h in range(24):
    if h not in grp_hour.groups: continue
    grp = grp_hour.get_group(h)
    wr_h = (grp > 0).mean()
    print(f'  {h:>3}h  {len(grp):>7,}  {wr_h*100:>7.2f}%  {grp.sum():>+12.2f}  {grp.mean():>+9.2f}')

# ── 7. P&L by confidence decile (directional trades only)
conf_dir     = confidence[:-1][preds[:-1] != 0]
decile_edges = np.percentile(conf_dir, np.arange(0, 110, 10))

print(f'\n{"=" * 68}')
print('P&L BY CONFIDENCE DECILE  (directional trades only)')
print(f'  {"Decile":>7}  {"Conf range":>16}  {"Trades":>7}  {"WinRate":>8}  {"Avg P&L":>9}  {"Sum P&L":>12}')
for i in range(10):
    lo, hi = decile_edges[i], decile_edges[i + 1]
    mask   = (conf_dir >= lo) & (conf_dir < hi) if i < 9 else (conf_dir >= lo)
    tp     = trades_pnl[mask]
    if len(tp) == 0: continue
    print(f'  {i+1:>5}th  [{lo:.4f}, {hi:.4f})  {len(tp):>7,}  {(tp>0).mean()*100:>7.2f}%  {tp.mean():>+9.2f}  {tp.sum():>+12.2f}')

# ── 8. After-fee PnL summary
print(f'\n{"=" * 68}')
print('FEE IMPACT')
print(f'  {"":>30}  {"Total P&L":>12}  {"Per trade":>10}')
print(f'  {"Zero fees":>30}  {strat_pnl:>+12,.2f}  {strat_pnl/max(n_trades,1):>+10.4f}')
print(f'  {"After maker (0.02%/side)":>30}  {strat_pnl - n_trades*maker_fee:>+12,.2f}  {(strat_pnl - n_trades*maker_fee)/max(n_trades,1):>+10.4f}')
print(f'  {"After taker (0.05%/side)":>30}  {strat_pnl - n_trades*taker_fee:>+12,.2f}  {(strat_pnl - n_trades*taker_fee)/max(n_trades,1):>+10.4f}')
needed_wr = 0.5 + maker_fee / (avg_win - avg_loss + 1e-9) / 2 if (avg_win - avg_loss) > 0 else float('nan')
print(f'  Breakeven WR (maker)   : {needed_wr*100:.2f}%  (current: {win_rate*100:.2f}%)')


In [ ]:
# ── Cell 10: Plots ────────────────────────────────────────────────────────
# Requires Cell 9b to have been run (defines cm, cm_pct, drawdown, sharpe,
# max_dd, pnl_by_hour, dt_trade_idx).

DARK, PANEL, GRID = '#0d1117', '#161b22', '#21262d'
TEXT, UP, DOWN, ACC = '#e6edf3', '#3fb950', '#f85149', '#58a6ff'
GOLD = '#f0883e'

fig = plt.figure(figsize=(18, 24), facecolor=DARK)
gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.45, wspace=0.30,
                        left=0.07, right=0.97, top=0.95, bottom=0.05)

def style(ax, title):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=TEXT, labelsize=8)
    ax.set_title(title, color=TEXT, fontsize=9, fontweight='bold', pad=6)
    for sp in ax.spines.values(): sp.set_edgecolor(GRID)
    ax.grid(color=GRID, linewidth=0.5, alpha=0.7)
    ax.yaxis.label.set_color(TEXT)
    ax.xaxis.label.set_color(TEXT)

ep = range(1, len(train_losses) + 1)

# Row 1 left: Training / val loss
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(ep, train_losses, color=ACC, label='Train loss')
ax1.plot(ep, val_losses,   color=GOLD, label='Val loss')
ax1.legend(facecolor=PANEL, labelcolor=TEXT, edgecolor=GRID, fontsize=8)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
style(ax1, 'Training / Validation Loss')

# Row 1 right: Val accuracy
ax1b = fig.add_subplot(gs[0, 1])
ax1b.plot(ep, [v*100 for v in val_accs], color=UP)
ax1b.axhline(dummy_acc*100, color=GRID, linestyle='--',
             label=f'Dummy baseline ({dummy_acc*100:.1f}%)')
ax1b.legend(facecolor=PANEL, labelcolor=TEXT, edgecolor=GRID, fontsize=8)
ax1b.set_xlabel('Epoch'); ax1b.set_ylabel('Accuracy (%)')
style(ax1b, 'Validation Accuracy')

# Row 2: Cumulative P&L (full width)
ax2 = fig.add_subplot(gs[1, :])
cum_strat = np.cumsum(trade_pnl)
cum_bnh   = price_test[1:] - price_test[0]
x_idx     = np.arange(len(cum_strat))
ax2.plot(x_idx, cum_strat, color=ACC, linewidth=1.2, label='LSTM v2 (zero fees)')
ax2.plot(x_idx, cum_bnh,   color=GOLD, linewidth=1.0, linestyle='--', label='Buy & Hold')
ax2.axhline(0, color=GRID, linewidth=0.8)
ax2.fill_between(x_idx, cum_strat, 0, where=cum_strat >= 0, alpha=0.12, color=UP)
ax2.fill_between(x_idx, cum_strat, 0, where=cum_strat <  0, alpha=0.12, color=DOWN)
ax2.set_ylabel('Cumulative P&L (USD)')
ax2.legend(facecolor=PANEL, labelcolor=TEXT, edgecolor=GRID, fontsize=9)
ax2.set_title(
    f'Backtest: ${strat_pnl:+,.0f} vs B&H ${bnh_pnl:+,.0f}  |  '
    f'WR {win_rate*100:.2f}%  |  {n_trades:,} trades  |  '
    f'Sharpe={sharpe:.2f}  Calmar={calmar:.2f}  MaxDD=${max_dd:,.0f}',
    color=TEXT, fontsize=9, fontweight='bold')
for sp in ax2.spines.values(): sp.set_edgecolor(GRID)
ax2.set_facecolor(PANEL); ax2.tick_params(colors=TEXT)
ax2.grid(color=GRID, linewidth=0.5, alpha=0.7)

# Row 3 left: Drawdown
ax3 = fig.add_subplot(gs[2, 0])
ax3.fill_between(x_idx, drawdown, 0, alpha=0.6, color=DOWN)
ax3.plot(x_idx, drawdown, color=DOWN, linewidth=0.7)
ax3.axhline(max_dd, color=GRID, linestyle=':', linewidth=0.8,
            label=f'Max DD ${max_dd:,.0f}')
ax3.legend(facecolor=PANEL, labelcolor=TEXT, edgecolor=GRID, fontsize=8)
ax3.set_ylabel('Drawdown (USD)')
style(ax3, f'Drawdown Curve  (max ${max_dd:,.0f}  =  {max_dd_pct:.1f}% of peak)')

# Row 3 right: Confusion matrix heatmap
ax4 = fig.add_subplot(gs[2, 1])
im = ax4.imshow(cm_pct, cmap='Blues', vmin=0, vmax=100, aspect='auto')
ax4.set_xticks([0, 1, 2]); ax4.set_yticks([0, 1, 2])
ax4.set_xticklabels(['DOWN', 'FLAT', 'UP'], color=TEXT)
ax4.set_yticklabels(['DOWN', 'FLAT', 'UP'], color=TEXT)
ax4.set_xlabel('Predicted', color=TEXT); ax4.set_ylabel('True', color=TEXT)
for i in range(3):
    for j in range(3):
        ax4.text(j, i, f'{cm[i,j]:,}\n({cm_pct[i,j]:.1f}%)',
                 ha='center', va='center', fontsize=9,
                 color='white' if cm_pct[i, j] > 55 else TEXT)
cb = plt.colorbar(im, ax=ax4)
cb.ax.tick_params(colors=TEXT); cb.set_label('% of true class', color=TEXT)
ax4.set_facecolor(PANEL)
ax4.tick_params(colors=TEXT, labelsize=9)
ax4.set_title('Confusion Matrix (% of true class)', color=TEXT, fontsize=9, fontweight='bold', pad=6)
for sp in ax4.spines.values(): sp.set_edgecolor(GRID)
ax4.grid(False)

# Row 4 left: Win rate vs confidence threshold
ax5 = fig.add_subplot(gs[3, 0])
threshs = np.arange(0.33, 0.81, 0.01)
wr_curve, n_curve = [], []
for t in threshs:
    mask = (confidence[:-1] >= t) & (preds[:-1] != 0)
    if mask.sum() < 10:
        wr_curve.append(np.nan); n_curve.append(0)
    else:
        tp = trade_pnl[mask]
        wr_curve.append((tp > 0).mean() * 100)
        n_curve.append(mask.sum())
ax5.plot(threshs * 100, wr_curve, color=ACC, linewidth=2)
ax5.axhline(50, color=GRID, linestyle='--', linewidth=0.8)
ax5.axhline(52, color=UP, linestyle=':', linewidth=0.8, label='~52% maker b/e')
ax5.set_xlabel('Confidence threshold (%)'); ax5.set_ylabel('Win rate (%)')
ax5.legend(facecolor=PANEL, labelcolor=TEXT, edgecolor=GRID, fontsize=8)
ax5b = ax5.twinx()
ax5b.bar(threshs * 100, n_curve, width=0.8, alpha=0.25, color=GOLD)
ax5b.set_ylabel('# trades', color=GOLD)
ax5b.tick_params(colors=GOLD, labelsize=7)
style(ax5, 'Win Rate vs Confidence Threshold')

# Row 4 right: Hourly P&L bar chart
ax6 = fig.add_subplot(gs[3, 1])
hours_h = pnl_by_hour.index.values
vals_h  = pnl_by_hour.values
colors_h = [UP if v >= 0 else DOWN for v in vals_h]
ax6.bar(hours_h, vals_h, color=colors_h, alpha=0.85, width=0.7)
ax6.axhline(0, color=GRID, linewidth=0.8)
ax6.set_xlabel('Hour (UTC)'); ax6.set_ylabel('Sum P&L (USD)')
ax6.set_xticks(range(0, 24, 2))
style(ax6, 'P&L by Hour of Day (UTC)')

fig.suptitle(
    f'LSTM v2 — HIDDEN={HIDDEN_DIM}  LAYERS={NUM_LAYERS}  SEQ={SEQ_LEN}b  ATTN  |  '
    f'Params={sum(p.numel() for p in model.parameters()):,}  |  '
    f'PF={profit_factor:.2f}  Sharpe={sharpe:.2f}',
    color=TEXT, fontsize=12, fontweight='bold')
plt.savefig(os.path.join(OUTPUT_DIR, 'lstm_v2_results.png'), dpi=150, bbox_inches='tight', facecolor=DARK)
plt.show()
print('Chart saved.')


In [ ]:
# ── Cell 11: Save model ───────────────────────────────────────────────────
save_path = os.path.join(OUTPUT_DIR, f'lstm_v2_t{THRESHOLD:.0f}_h{HORIZON}.pt')
torch.save({
    'model_state':   model.state_dict(),
    'scaler':        scaler,
    'features':      features,
    'seq_len':       SEQ_LEN,
    'hidden_dim':    HIDDEN_DIM,
    'num_layers':    NUM_LAYERS,
    'nhead':         NHEAD,
    'dropout':       DROPOUT,
    'threshold':     THRESHOLD,
    'horizon':       HORIZON,
    'val_acc':       max(val_accs),
    'test_win_rate': win_rate,
    'n_test_trades': n_trades,
}, save_path)
print(f'Model saved → {save_path}')
print(f'\nSummary:')
print(f'  Architecture  : LSTM({HIDDEN_DIM}×{NUM_LAYERS}) + MHA({NHEAD} heads)')
print(f'  Parameters    : {sum(p.numel() for p in model.parameters()):,}')
print(f'  Features      : {len(features)}')
print(f'  Sequence      : {SEQ_LEN} bars ({SEQ_LEN*WINDOW_SEC}s)')
print(f'  Val accuracy  : {max(val_accs)*100:.2f}%')
print(f'  Test win rate : {win_rate*100:.2f}%  ({n_trades:,} trades)')
print(f'  Strategy P&L  : ${strat_pnl:+,.2f}  (zero fees)')
print(f'  After maker   : ${strat_pnl - n_trades*maker_fee:+,.2f}')

In [ ]:
# ── Cell 12: Load saved model ─────────────────────────────────────────────
#
# checkpoint = torch.load(save_path, map_location=device)
# features   = checkpoint['features']
# scaler     = checkpoint['scaler']
# model      = BtcLSTMAttn(
#     input_dim=len(features),
#     hidden_dim=checkpoint['hidden_dim'],
#     num_layers=checkpoint['num_layers'],
#     nhead=checkpoint['nhead'],
#     dropout=checkpoint['dropout'],
# ).to(device)
# model.load_state_dict(checkpoint['model_state'])
# model.eval()
# print('Model loaded. Val acc:', checkpoint['val_acc']*100, '%')
print('Uncomment the block above to load a saved model.')